# shelf-vision: inference tuning + WBF ensemble (no training)

Two existing YOLO11m checkpoints: **base** (960-trained, test mAP50-95 0.612) and
**pl** (pseudo-label fine-tune at 1280).

1. On **val**: grid over model × imgsz × TTA × NMS IoU (conf 0.001-ish, max_det 1000).
2. On **val**: greedy WBF ensemble over those configs, then WBF IoU threshold.
3. On **test**: only the chosen configs, plus the untuned baselines, scored once.

Everything is scored with one evaluator that mirrors ultralytics' matching and
101-point AP, sanity-checked against `model.val()` on val.

Predictions are made once per (model, imgsz, TTA) at NMS IoU 0.8; lower IoUs are
re-applied with torchvision NMS on top (equivalent up to rare chain-suppression
edge cases).

**Inputs:** notebooks `shelf-vision-yolo11m` and `shelf-vision-yolo11m-pl`.
GPU T4, Internet on.

In [ ]:
import glob, os, json, time, itertools, pickle, subprocess
from pathlib import Path
import numpy as np

DATA_DIR = "/tmp/shelf-data"
os.environ["SHELF_DATA_DIR"] = DATA_DIR
CFG = "shelf-vision/configs/detect/yolo11m-sku110k.yaml"   # original labels for val/test

base = sorted(glob.glob("/kaggle/input/**/yolo11m-sku110k-*h/weights/best.pt", recursive=True))
pl = sorted(glob.glob("/kaggle/input/**/yolo11m-sku110k-pl/weights/best.pt", recursive=True))
assert base and pl, (base, pl)
WEIGHTS = {"base": base[-1], "pl": pl[-1]}
print(WEIGHTS)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q -b pseudo-labels https://github.com/rsuvbe/shelf-vision
!git -C shelf-vision log --oneline -1
!pip install -q -e shelf-vision ensemble-boxes

In [ ]:
!bash shelf-vision/scripts/fetch_sku110k.sh {DATA_DIR}
out = subprocess.run(["shelf", "prepare", CFG], capture_output=True, text=True, check=True).stdout
print(out[-2000:])
DATA = out.strip().splitlines()[-1]
!rm -f {DATA_DIR}/SKU110K_fixed.tar.gz
from ultralytics.data.utils import check_det_dataset
DS = check_det_dataset(DATA)

def split_paths(split):
    lst = Path(DS[split])
    root = Path(DS["path"])
    paths = []
    for line in lst.read_text().splitlines():
        line = line.strip()
        if line:
            p = Path(line)
            paths.append(str(p if p.is_absolute() else (root / p).resolve()))
    return paths

def load_gt(paths):
    gt = {}
    for p in paths:
        lp = Path(p.replace("/images/", "/labels/")).with_suffix(".txt")
        a = np.loadtxt(lp, ndmin=2) if lp.exists() and lp.stat().st_size else np.zeros((0, 5))
        xc, yc, w, h = a[:, 1], a[:, 2], a[:, 3], a[:, 4]
        gt[Path(p).stem] = np.stack([xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2], 1).astype(np.float32)
    return gt

PATHS = {s: split_paths(s) for s in ("val", "test")}
GT = {s: load_gt(PATHS[s]) for s in PATHS}
for s in PATHS:
    print(s, len(PATHS[s]), "images", sum(len(v) for v in GT[s].values()), "boxes")

## Evaluator, prediction, NMS, WBF

In [ ]:
import torch, torchvision
from ultralytics import YOLO

IOUV = np.linspace(0.5, 0.95, 10)

def _match(pred, gt):
    """COCO-style matching as in ultralytics>=8.4: detections in descending confidence
    each claim the best still-unclaimed GT, per IoU threshold. pred/gt: xyxy."""
    tp = np.zeros((len(pred), len(IOUV)), bool)
    if not len(pred) or not len(gt):
        return tp
    order = np.argsort(-pred[:, 4], kind="stable")
    iou = torchvision.ops.box_iou(torch.from_numpy(gt), torch.from_numpy(pred[order, :4])).numpy()
    matched = np.zeros((len(gt), len(IOUV)), bool)
    cols = np.arange(len(IOUV))
    for j in np.flatnonzero((iou >= IOUV.min()).any(0)):
        avail = np.where(matched, 0, iou[:, j, None])
        k = avail.argmax(0)
        ok = avail[k, cols] >= IOUV
        tp[order[j]] = ok
        matched[k, cols] |= ok
    return tp

from ultralytics.utils.metrics import compute_ap  # same AP as model.val()

def _ap(rec, prec):
    return compute_ap(rec, prec)[0]

def evaluate(preds, split):
    """preds: {stem: (n,5) xyxy-normalised + conf}. Returns (mAP50-95, mAP50)."""
    gt = GT[split]
    tps, confs, n_gt = [], [], 0
    for stem, g in gt.items():
        p = preds.get(stem, np.zeros((0, 5), np.float32))
        tps.append(_match(p, g)); confs.append(p[:, 4]); n_gt += len(g)
    tp = np.concatenate(tps); conf = np.concatenate(confs)
    tp = tp[np.argsort(-conf, kind="stable")]
    tpc = tp.cumsum(0); fpc = (1 - tp).cumsum(0)
    rec = tpc / (n_gt + 1e-16); prec = tpc / (tpc + fpc)
    ap = np.array([_ap(rec[:, j], prec[:, j]) for j in range(len(IOUV))])
    return float(ap.mean()), float(ap[0])

PRED_DIR = Path("/tmp/preds"); PRED_DIR.mkdir(exist_ok=True)
CONF, MAX_DET = 0.001, 1000

def predict(model_name, imgsz, tta, split):
    key = f"{split}_{model_name}_{imgsz}_{'tta' if tta else 'plain'}"
    f = PRED_DIR / f"{key}.pkl"
    if f.exists():
        return pickle.loads(f.read_bytes())
    t = time.time()
    model = YOLO(WEIGHTS[model_name])
    out = {}
    # a .txt source streams images from disk; a Python list is opened all at once
    # by ultralytics (PIL), which is ~20 GB for val and kills the kernel
    src = PRED_DIR / f"{split}_paths.txt"
    src.write_text("\n".join(PATHS[split]) + "\n")
    for r in model.predict(str(src), stream=True, imgsz=imgsz, conf=CONF, iou=0.8,
                           max_det=MAX_DET, augment=tta, half=True, device=0,
                           batch=4 if tta else 8, verbose=False):
        h, w = r.orig_shape
        b = r.boxes.xyxy.cpu().numpy().astype(np.float32)
        b[:, [0, 2]] /= w; b[:, [1, 3]] /= h
        out[Path(r.path).stem] = np.concatenate([b.clip(0, 1), r.boxes.conf.cpu().numpy()[:, None]], 1)
    f.write_bytes(pickle.dumps(out))
    print(f"  predicted {key} in {time.time() - t:.0f}s")
    del model; torch.cuda.empty_cache()
    return out

def renms(preds, iou):
    if iou >= 0.8:
        return preds
    out = {}
    for k, p in preds.items():
        if len(p):
            keep = torchvision.ops.nms(torch.from_numpy(p[:, :4]), torch.from_numpy(p[:, 4]), iou).numpy()
            p = p[keep[:MAX_DET]]
        out[k] = p
    return out

In [ ]:
from multiprocessing import get_context
from ensemble_boxes import weighted_boxes_fusion

_MEMBERS = None

def _wbf_one(args):
    stem, iou_thr = args
    boxes = [m[stem][:, :4].tolist() if stem in m and len(m[stem]) else [] for m in _MEMBERS]
    scores = [m[stem][:, 4].tolist() if stem in m and len(m[stem]) else [] for m in _MEMBERS]
    labels = [[0] * len(s) for s in scores]
    if not any(scores):
        return stem, np.zeros((0, 5), np.float32)
    b, s, _ = weighted_boxes_fusion(boxes, scores, labels, iou_thr=iou_thr, skip_box_thr=CONF)
    o = np.argsort(-s)[:MAX_DET]
    return stem, np.concatenate([b[o], s[o, None]], 1).astype(np.float32)

def wbf(members, iou_thr, split):
    global _MEMBERS
    _MEMBERS = members
    with get_context("fork").Pool(os.cpu_count()) as pool:  # WBF is CPU-only
        res = pool.map(_wbf_one, [(s, iou_thr) for s in GT[split]], chunksize=8)
    return dict(res)

## Sanity check: our evaluator vs `model.val()` (pl @ 1280, val, NMS IoU 0.7)

In [ ]:
m = YOLO(WEIGHTS["pl"]).val(data=DATA, split="val", imgsz=1280, conf=CONF, iou=0.7, max_det=MAX_DET,
                            device=0, half=True, plots=False, verbose=False)
ours = evaluate(renms(predict("pl", 1280, False, "val"), 0.7), "val")
print(f"ultralytics val: mAP50-95 {m.box.map:.4f}  mAP50 {m.box.map50:.4f}")
print(f"ours           : mAP50-95 {ours[0]:.4f}  mAP50 {ours[1]:.4f}")
torch.cuda.empty_cache()

## 1. Val grid: model × imgsz × TTA × NMS IoU

In [ ]:
CONFIGS = [(m, s, t) for m in ("base", "pl")
           for s, t in [(1280, False), (1536, False), (1792, False), (1280, True), (1536, True)]]
NMS_IOUS = [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8]

def cname(c):
    return f"{c[0]}@{c[1]}{'+tta' if c[2] else ''}"

grid, best_iou = [], {}
for c in CONFIGS:
    p = predict(*c, "val")
    scores = {iou: evaluate(renms(p, iou), "val") for iou in NMS_IOUS}
    bi = max(scores, key=lambda i: scores[i][0])
    best_iou[c] = bi
    for iou, (m95, m50) in scores.items():
        grid.append(dict(config=cname(c), nms_iou=iou, map=m95, map50=m50))
    print(f"{cname(c):16} " + "  ".join(f"{i}:{scores[i][0]:.4f}" for i in NMS_IOUS) + f"   best iou {bi}")

single = sorted(((evaluate(renms(predict(*c, "val"), best_iou[c]), "val")[0], c) for c in CONFIGS), reverse=True)
print("\nbest single configs on val:")
for s, c in single[:5]:
    print(f"  {cname(c):16} iou {best_iou[c]}  mAP50-95 {s:.4f}")

## 2. Val: greedy WBF ensemble, then WBF IoU threshold

In [ ]:
VAL_MEMBER = {c: renms(predict(*c, "val"), best_iou[c]) for c in CONFIGS}
best_score, best_c = single[0]
selected = [best_c]
ens_log = []
for step in range(3):
    trials = []
    for c in CONFIGS:
        if c in selected:
            continue
        t = time.time()
        s = evaluate(wbf([VAL_MEMBER[x] for x in selected + [c]], 0.55, "val"), "val")[0]
        trials.append((s, c))
        print(f"  step {step+1}: + {cname(c):16} -> {s:.4f}  ({time.time()-t:.0f}s)")
    s, c = max(trials)
    ens_log.append(dict(step=step + 1, add=cname(c), map=s))
    if s < best_score + 0.0005:
        print(f"no gain from step {step+1}, stop")
        break
    selected.append(c); best_score = s
    print(f"=> ensemble {[cname(x) for x in selected]}  mAP50-95 {s:.4f}")

wbf_iou, wbf_scores = 0.55, {}
if len(selected) > 1:
    for thr in [0.45, 0.5, 0.55, 0.6, 0.65, 0.7]:
        wbf_scores[thr] = evaluate(wbf([VAL_MEMBER[x] for x in selected], thr, "val"), "val")
        print(f"  wbf iou_thr {thr}: mAP50-95 {wbf_scores[thr][0]:.4f}  mAP50 {wbf_scores[thr][1]:.4f}")
    wbf_iou = max(wbf_scores, key=lambda t: wbf_scores[t][0])
print("chosen:", [cname(x) for x in selected], "wbf iou", wbf_iou)

## 3. Test: baselines, best single, ensemble (each scored once)

In [ ]:
test_rows = []
def add(name, preds, note=""):
    m95, m50 = evaluate(preds, "test")
    test_rows.append(dict(name=name, map=m95, map50=m50, note=note))
    print(f"{name:40} mAP50-95 {m95:.4f}  mAP50 {m50:.4f}  {note}")

add("base@960 iou0.7 (original)", renms(predict("base", 960, False, "test"), 0.7))
add("base@1280 iou0.7", renms(predict("base", 1280, False, "test"), 0.7))
add("pl@1280 iou0.7", renms(predict("pl", 1280, False, "test"), 0.7))
bc = single[0][1]
add(f"best single: {cname(bc)} iou{best_iou[bc]}", renms(predict(*bc, "test"), best_iou[bc]), "tuned on val")
if len(selected) > 1:
    members = [renms(predict(*c, "test"), best_iou[c]) for c in selected]
    add("WBF " + " + ".join(cname(c) for c in selected) + f" thr{wbf_iou}",
        wbf(members, wbf_iou, "test"), "tuned on val")

res = dict(val_grid=grid, best_nms_iou={cname(c): i for c, i in best_iou.items()},
           ensemble_log=ens_log, ensemble=[cname(c) for c in selected], wbf_iou=wbf_iou,
           wbf_iou_scores={str(k): v for k, v in wbf_scores.items()}, test=test_rows)
Path("/kaggle/working/tune_results.json").write_text(json.dumps(res, indent=2))
print("\nsaved /kaggle/working/tune_results.json")